# Level 1 — is the interpolated 9-point predictive CDF calibrated at every 5% rung?

The product card prints **"~C% chance this day will be `<predicate>`"**. C is an
integral of the forecast's **9-point CQR-calibrated band** treated as a
piecewise-linear predictive CDF. As of the q9 retrain **all 9 points are trained
quantile heads** (levels .01/.05/.10/.25/.50/.75/.90/.95/.99) — the interior
q25/q75 used to be probit-interpolated client-side and now come off the table.
The card's `invCdf` still reads values at arbitrary 5% rungs that are themselves
linear interpolations *between* those 9 points.

**Question (Level 1 of the validation relay):** build the forecast band exactly
as the frontend does, and ask — is that piecewise-linear CDF calibrated at every
5% rung, especially the **12 interpolated rungs**
(15,20,30,35,40,45,55,60,65,70,80,85)? Rungs 25 and 75 moved from interpolated
to trained in the q9 retrain, so they are now Level-0 anchors, not news.

**Honest split.** The held-out **test** month-blocks are cut into two disjoint
halves — *calib* blocks fit the per-level CQR shifts (exactly like production),
*eval* blocks measure coverage. Neither the CatBoost fit nor the CQR shifts ever
saw the eval blocks, so rung coverage there is an out-of-sample read. Level 0
(`cqr_per_level_validity.ipynb`) already showed the 9 trained heads cover at
their nominal levels; the 12 interpolated rungs are the news here.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from catboost import CatBoostRegressor
import train_quantile_debias as tqd
import make_debias_tables as mdt   # reuse the PRODUCTION per-level CQR shift + isotonic

pd.set_option("display.width", 170, "display.float_format", lambda v: f"{v:.3f}")

TAG = "qn8727_s0_q9"
QUANTILE_LEVELS = tqd.QUANTILES                  # [.01 .05 .10 .25 .50 .75 .90 .95 .99]
I50 = QUANTILE_LEVELS.index(0.50)
# services/tieredData.ts / confidence.ts PRECIP_TRACE_MM — a value <1mm reads as 0mm.
PRECIP_TRACE_MM = 1.0

# The 9 CDF points the frontend integrates — now ALL trained heads, so the band
# columns map 1:1 onto QUANTILE_LEVELS in order (asserted, since the whole
# notebook indexes one by the other).
BAND_P = np.array([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
BAND_COLS = ["q01", "lo", "q10", "q25", "mid", "q75", "q90", "hi", "q99"]
assert np.allclose(BAND_P, QUANTILE_LEVELS), "band points must equal the trained levels"

# rung ladder the card reads off the CDF; which of those are TRAINED heads.
RUNGS = list(range(5, 100, 5))                               # 5,10,...,95
TRAINED_RUNGS = {5, 10, 25, 50, 75, 90, 95}                  # multiples of 5 that are trained
INTERP_RUNGS = [r for r in RUNGS if r not in TRAINED_RUNGS]  # the 12 interpolated rungs

# cache is keyed by TAG: the 7-level bands from the previous run live alongside
# these and MUST NOT be picked up by a skip-if-exists load.
CACHE_DIR = Path("data/nb_confidence"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("tag:", TAG)
print("quantile levels:", QUANTILE_LEVELS)
print("9 CDF points p =", BAND_P.tolist())
print(f"trained rungs: {sorted(TRAINED_RUNGS)}")
print(f"interpolated rungs ({len(INTERP_RUNGS)}):", INTERP_RUNGS)

quantile levels: [0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99]
9 CDF points p = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
interpolated rungs (14): [15, 20, 25, 30, 35, 40, 45, 55, 60, 65, 70, 75, 80, 85]


In [2]:
# --- load the cached post-split processed frame (features + split already baked) ---
frame = pd.read_feather("data/pipeline_frame_full.feather")
print(f"{len(frame):,} rows, {frame.key.nunique()} cells, "
      f"{frame.date.min().date()}..{frame.date.max().date()}")

# GUARD: a cached frame built before the fc_version mask-order fix carries WRONG
# cycle labels (the 49r1 mask overwrote every 50r1 row), and a model trained with
# the fix would then be evaluated on data the fix never touched. Recompute the
# labels from the frame's own dates and refuse a frame that disagrees.
def assert_fc_version_current(frame):
    expected = pd.Series(tqd.FC_BASE, index=frame.index)
    for change_date, cycle in sorted(tqd.FC_CHANGES):
        expected = expected.mask(frame["date"] >= change_date, cycle)
    actual = frame["fc_version"].astype(str)
    bad = int((actual != expected.astype(str)).sum())
    assert bad == 0, (
        f"cached frame has STALE fc_version labels ({bad:,} rows disagree; "
        f"cached={dict(actual.value_counts())}). Delete "
        f"data/pipeline_frame_full.feather and rebuild from source."
    )
    print("fc_version guard OK:", dict(actual.value_counts()))

assert_fc_version_current(frame)

# monthly block id exactly as tqd.split builds it, so the TEST blocks split into
# disjoint calib vs eval halves (copied from q25_q75_interp_check.ipynb cell 2).
origin = frame.date.min().normalize()
frame["block"] = ((frame.date.dt.year - origin.year) * 12
                  + (frame.date.dt.month - origin.month))
test_blocks = np.sort(frame.loc[frame.role == "test", "block"].unique())
calib_blocks = set(test_blocks[0::2])                        # even-indexed -> calib
eval_blocks  = set(test_blocks[1::2])                        # odd-indexed  -> eval
print(f"test blocks: {list(test_blocks)}")
print(f"  calib -> {sorted(calib_blocks)}")
print(f"  eval  -> {sorted(eval_blocks)}")

7,179,885 rows, 8727 cells, 2024-03-01..2026-06-01


test blocks: [np.int32(0), np.int32(5), np.int32(10), np.int32(15), np.int32(20), np.int32(25)]
  calib -> [np.int32(0), np.int32(10), np.int32(20)]
  eval  -> [np.int32(5), np.int32(15), np.int32(25)]


In [3]:
# --- exact ports of the frontend CDF primitives (port correctness is load-bearing
#     for this notebook AND Notebook 2, so cell 5 asserts them) ------------------

def precip_trace_clamp(values):
    "confidence.ts bandQuantilePoints / tieredData: a value <1mm reads as 0mm."
    values = np.asarray(values, float)
    return np.where(values < PRECIP_TRACE_MM, 0.0, values)

def value_at_tail_fraction(values, q, is_high_side):
    "confidence.ts:122 order-statistic boundary; sorts a copy, clamps the index."
    sorted_v = np.sort(np.asarray(values, float))
    n = len(sorted_v)
    idx = int(np.floor((1 - q) * n)) if is_high_side else int(np.ceil(q * n)) - 1
    return float(sorted_v[min(max(idx, 0), n - 1)])

def prob_in_interval(point_v, point_p, lo, hi):
    "confidence.ts:70 — mass of the 9-point CDF in [lo,hi]; point masses + outer tails."
    v = np.asarray(point_v, float); p = np.asarray(point_p, float)
    n = len(v); total = 0.0
    if lo <= v[0]  <= hi: total += p[0]                      # mass below p01
    if lo <= v[-1] <= hi: total += 1 - p[-1]                 # mass above p99
    for i in range(n - 1):
        va, vb = v[i], v[i + 1]; dp = p[i + 1] - p[i]
        if vb == va:
            if lo <= va <= hi: total += dp                   # point mass at va
        else:
            lo_c = min(max(lo, va), vb); hi_c = min(max(hi, va), vb)
            total += dp * ((hi_c - lo_c) / (vb - va))
    return float(min(max(total, 0.0), 1.0))

def inv_cdf_col(band, p_points, u):
    "HistogramChart.invCdf, vectorized over rows for one probability u. band: (n,9)."
    if u <= p_points[0]:  return band[:, 0].copy()
    if u >= p_points[-1]: return band[:, -1].copy()
    i = int(np.searchsorted(p_points, u, side="right") - 1)  # p[i] <= u < p[i+1]
    pa, pb = p_points[i], p_points[i + 1]
    va, vb = band[:, i], band[:, i + 1]
    return va.copy() if pb == pa else va + ((u - pa) / (pb - pa)) * (vb - va)

In [4]:
# --- port sanity: hand-built cases, assert (do not eyeball) ---------------------
non_degenerate = np.array([0., 1, 2, 3, 4, 5, 6, 7, 8])     # a strictly-rising band
# 1. prob_in_interval: fully enclosed -> ~1.0 (outer tails counted, not 0.98)
assert abs(prob_in_interval(non_degenerate, BAND_P, -1, 9) - 1.0) < 1e-9
# an interior bracket [mid, q90] == [4, 6] spans p .50->.90 = .40 mass
assert abs(prob_in_interval(non_degenerate, BAND_P, 4, 6) - 0.40) < 1e-9
# one-sided high tail [q90, inf) -> 1 - .90 = .10
assert abs(prob_in_interval(non_degenerate, BAND_P, 6, np.inf) - 0.10) < 1e-9
# 2. point-mass spike: a bone-dry precip band (all zeros)
dry = np.zeros(9)
assert abs(prob_in_interval(dry, BAND_P, -1, 1) - 1.0) < 1e-9    # 0 inside  -> ~1
assert abs(prob_in_interval(dry, BAND_P, 0.5, 5) - 0.0) < 1e-9   # 0 excluded -> ~0
# 3. value_at_tail_fraction: behavioural check against the pool it slices
rng = np.random.default_rng(0); pool = rng.normal(size=5000)
for q in (0.05, 0.10, 0.20):
    hi_b = value_at_tail_fraction(pool, q, True)
    lo_b = value_at_tail_fraction(pool, q, False)
    assert abs(np.mean(pool >= hi_b) - q) < 1e-3    # ~q of the pool at/above the high cut
    assert abs(np.mean(pool <= lo_b) - q) < 1e-3    # ~q of the pool at/below the low cut
# 4. inv_cdf recovers the 9 anchors exactly and interpolates the interior linearly
band1 = non_degenerate.reshape(1, -1)
for j, u in enumerate(BAND_P):
    assert abs(inv_cdf_col(band1, BAND_P, u)[0] - non_degenerate[j]) < 1e-9
expect_15 = 2 + (0.15 - 0.10) / (0.25 - 0.10) * (3 - 2)     # between q10(=2) and q25(=3)
assert abs(inv_cdf_col(band1, BAND_P, 0.15)[0] - expect_15) < 1e-9
print("port sanity: all asserts passed")

port sanity: all asserts passed


In [5]:
# --- per-var calibrated 9-point band on EVAL rows, cached for reuse. Notebook 2
#     reads these feathers and never touches the models. Skip-if-exists. ---------

GATED = mdt.gated_keys(TAG, 0.1)          # {var -> set(cell keys whose ML band is gated out)}

def predict_sorted(model, X, chunk=200_000):
    "chunked 9-head predict (tmax model is ~400MB); sort heads per row as production does."
    out = [np.sort(np.asarray(model.predict(X.iloc[s:s + chunk])), axis=1)
           for s in range(0, len(X), chunk)]
    return np.concatenate(out, axis=0)

def build_eval_bands(name):
    "eval-row frame (key,date,hres,truth_abs,gated + 9 absolute band cols)."
    cache = CACHE_DIR / f"eval_bands_{name}_{TAG}.feather"
    if cache.exists():
        print(f"[{name}] load cache {cache.name}")
        return pd.read_feather(cache)

    var = next(v for v in tqd.VARS if v["name"] == name)
    nonneg = var["nonneg"]; hcol = tqd.hres_col(name)
    feat = tqd.feature_cols(name, with_cell=True, with_cross=False)
    rows = frame[(frame.role == "test") & frame[f"bias_{name}"].notna()]
    X = rows[feat].astype({"key": str, "fc_version": str})

    model = CatBoostRegressor(); model.load_model(str(tqd.MODELS / f"M3_base_{name}_{TAG}.cbm"))
    preds = predict_sorted(model, X)                          # (n,9) sorted bias-delta heads
    del model

    y    = rows[f"bias_{name}"].to_numpy()
    hres = rows[hcol].to_numpy()
    is_calib = rows.block.isin(calib_blocks).to_numpy()
    is_eval  = rows.block.isin(eval_blocks).to_numpy()

    # per-level CQR shift fit on CALIB only, median head stays 0 (mirror compute_level_shifts)
    shifts = np.zeros(len(QUANTILE_LEVELS))
    for i, level in enumerate(QUANTILE_LEVELS):
        if i != I50:
            shifts[i] = mdt.per_level_shift(y[is_calib] - preds[is_calib, i], level)

    # apply to EVAL exactly as production bakes the tables: round 2dp, re-isotonize
    adj = mdt.pin_median_isotonic(np.round(preds[is_eval] + shifts, 2))    # (m,9) bias deltas
    abs_heads = hres[is_eval][:, None] + adj                               # -> absolute units
    if nonneg:
        abs_heads = np.maximum(abs_heads, 0.0)                             # ci.ts nonneg floor

    # all nine points are trained heads now — no shoulder interpolation step
    band = np.round(abs_heads, 3)                                          # ci.ts round3

    truth_abs = hres[is_eval] + y[is_eval]
    if name == "precip":                                                  # trace clamp band + truth
        band = precip_trace_clamp(band)
        truth_abs = precip_trace_clamp(truth_abs)

    ev_key = rows.key.to_numpy()[is_eval]
    out = pd.DataFrame({
        "key":       ev_key,
        "date":      rows.date.to_numpy()[is_eval],
        "hres":      hres[is_eval],
        "truth_abs": truth_abs,
        "gated":     pd.Series(ev_key).astype(str).isin(GATED[name]).to_numpy(),
    })
    for j, c in enumerate(BAND_COLS):
        out[c] = band[:, j]

    # the 9 points must stay ascending after clamps (the CDF must be valid)
    assert (np.diff(out[BAND_COLS].to_numpy(), axis=1) >= -1e-9).all(), f"{name}: band not monotone"
    out.to_feather(cache)
    print(f"[{name}] built {len(out):,} eval rows -> {cache.name} (gated {out.gated.mean()*100:.1f}%)")
    return out

eval_bands = {v["name"]: build_eval_bands(v["name"]) for v in tqd.VARS}
{k: len(df) for k, df in eval_bands.items()}

[20:58:22]   gate tmax: 56 / 8727 cells dropped (damage > 0.1)


[20:58:22]   gate tmin: 19 / 8727 cells dropped (damage > 0.1)


[20:58:22]   gate precip: 434 / 8727 cells dropped (damage > 0.1)


[20:58:22]   gate wind: 16 / 8727 cells dropped (damage > 0.1)


[tmax] built 794,157 eval rows -> eval_bands_tmax.feather (gated 0.6%)


[tmin] built 794,157 eval rows -> eval_bands_tmin.feather (gated 0.2%)


[precip] built 794,157 eval rows -> eval_bands_precip.feather (gated 5.0%)


[wind] built 794,157 eval rows -> eval_bands_wind.feather (gated 0.2%)


{'tmax': 794157, 'tmin': 794157, 'precip': 794157, 'wind': 794157}

In [6]:
# --- marginal rung coverage: empirical P(truth <= inv_cdf(band, r%)) vs r, in pp ---
def rung_coverage(df, rung_list=RUNGS):
    band = df[BAND_COLS].to_numpy(); truth = df.truth_abs.to_numpy()
    recs = []
    for r in rung_list:
        thr = inv_cdf_col(band, BAND_P, r / 100)
        cov = float(np.mean(truth <= thr))
        # flag rungs that land on a point-mass (flat) segment of the CDF
        i = min(max(int(np.searchsorted(BAND_P, r / 100, side="right") - 1), 0), len(BAND_P) - 2)
        flat_frac = float(np.mean(band[:, i + 1] - band[:, i] <= 1e-9))
        recs.append(dict(rung=r, kind=("trained" if r in TRAINED_RUNGS else "interp"),
                         cov_pct=round(cov * 100, 2), err_pp=round((cov - r / 100) * 100, 2),
                         flat_frac=round(flat_frac, 3)))
    return pd.DataFrame(recs)

rung_tables = {}
for name, df in eval_bands.items():
    tbl = rung_coverage(df); rung_tables[name] = tbl
    print(f"\n=== {name}: marginal rung coverage  (n={len(df):,}) ===")
    print(tbl.to_string(index=False))

# precip also on the WET-forecast subset (hres >= 1mm), where the point mass lifts
wet = eval_bands["precip"][eval_bands["precip"].hres >= 1.0]
rung_tables["precip_wet"] = rung_coverage(wet)
print(f"\n=== precip WET-forecast subset  (hres>=1mm, n={len(wet):,}) ===")
print(rung_tables["precip_wet"].to_string(index=False))


=== tmax: marginal rung coverage  (n=794,157) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained    4.580  -0.420      0.000
   10 trained    9.350  -0.650      0.000
   15  interp   12.930  -2.070      0.000
   20  interp   17.570  -2.430      0.000
   25  interp   23.250  -1.750      0.000
   30  interp   27.700  -2.300      0.000
   35  interp   32.610  -2.390      0.000
   40  interp   37.970  -2.030      0.000
   45  interp   43.580  -1.420      0.000
   50 trained   49.360  -0.640      0.000
   55  interp   55.090   0.090      0.000
   60  interp   60.690   0.690      0.000
   65  interp   66.050   1.050      0.000
   70  interp   70.990   0.990      0.000
   75  interp   75.500   0.500      0.000
   80  interp   81.330   1.330      0.000
   85  interp   86.050   1.050      0.000
   90 trained   89.750  -0.250      0.000
   95 trained   94.870  -0.130      0.000



=== tmin: marginal rung coverage  (n=794,157) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained    4.410  -0.590      0.000
   10 trained    9.330  -0.670      0.000
   15  interp   13.120  -1.880      0.000
   20  interp   18.000  -2.000      0.000
   25  interp   23.970  -1.030      0.000
   30  interp   28.540  -1.460      0.000
   35  interp   33.560  -1.440      0.000
   40  interp   38.980  -1.020      0.000
   45  interp   44.660  -0.340      0.000
   50 trained   50.430   0.430      0.000
   55  interp   56.160   1.160      0.000
   60  interp   61.720   1.720      0.000
   65  interp   66.990   1.990      0.000
   70  interp   71.860   1.860      0.000
   75  interp   76.300   1.300      0.000
   80  interp   82.060   2.060      0.000
   85  interp   86.700   1.700      0.000
   90 trained   90.340   0.340      0.000
   95 trained   95.240   0.240      0.000



=== precip: marginal rung coverage  (n=794,157) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained   58.830  53.830      0.825
   10 trained   59.630  49.630      0.684
   15  interp   60.660  45.660      0.684
   20  interp   62.280  42.280      0.684
   25  interp   64.510  39.510      0.632
   30  interp   65.990  35.990      0.632
   35  interp   67.450  32.450      0.632
   40  interp   68.970  28.970      0.632
   45  interp   70.640  25.640      0.632
   50 trained   72.330  22.330      0.455
   55  interp   75.330  20.330      0.455
   60  interp   78.080  18.080      0.455
   65  interp   80.770  15.770      0.455
   70  interp   83.170  13.170      0.455
   75  interp   85.250  10.250      0.319
   80  interp   87.760   7.760      0.319
   85  interp   89.700   4.700      0.319
   90 trained   91.530   1.530      0.190
   95 trained   95.440   0.440      0.057



=== wind: marginal rung coverage  (n=794,157) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained    5.140   0.140      0.000
   10 trained   10.170   0.170      0.000
   15  interp   14.100  -0.900      0.000
   20  interp   18.980  -1.020      0.000
   25  interp   24.790  -0.210      0.000
   30  interp   29.160  -0.840      0.000
   35  interp   33.910  -1.090      0.000
   40  interp   38.980  -1.020      0.000
   45  interp   44.290  -0.710      0.000
   50 trained   49.760  -0.240      0.000
   55  interp   55.340   0.340      0.000
   60  interp   60.790   0.790      0.000
   65  interp   66.010   1.010      0.000
   70  interp   70.900   0.900      0.000
   75  interp   75.370   0.370      0.000
   80  interp   81.220   1.220      0.000
   85  interp   86.110   1.110      0.000
   90 trained   89.970  -0.030      0.000
   95 trained   95.020   0.020      0.000



=== precip WET-forecast subset  (hres>=1mm, n=301,861) ===
 rung    kind  cov_pct  err_pp  flat_frac
    5 trained   13.930   8.930      0.540
   10 trained   16.040   6.040      0.172
   15  interp   18.740   3.740      0.172
   20  interp   23.000   3.000      0.172
   25  interp   28.860   3.860      0.046
   30  interp   32.740   2.740      0.046
   35  interp   36.570   1.570      0.046
   40  interp   40.540   0.540      0.046
   45  interp   44.870  -0.130      0.046
   50 trained   49.240  -0.760      0.001
   55  interp   57.000   2.000      0.001
   60  interp   63.850   3.850      0.001
   65  interp   69.800   4.800      0.001
   70  interp   74.700   4.700      0.001
   75  interp   78.760   3.760      0.000
   80  interp   83.700   3.700      0.000
   85  interp   87.380   2.380      0.000
   90 trained   90.170   0.170      0.000
   95 trained   95.210   0.210      0.000


In [7]:
# --- regime-conditional worst-rung error: within forecast-magnitude and band-width
#     quintiles (Gate 1: <=4pp in any bucket with n>=5000) -----------------------
def worst_rung_by_bucket(df, key_series, label):
    band = df[BAND_COLS].to_numpy(); truth = df.truth_abs.to_numpy()
    buckets = pd.qcut(key_series, 5, duplicates="drop")
    recs = []
    for bucket, idx in df.groupby(buckets, observed=True).indices.items():
        if len(idx) < 5000:                                   # Gate 1 judges n>=5000 buckets
            continue
        sub_band, sub_truth = band[idx], truth[idx]
        worst, at_rung = 0.0, None
        for r in RUNGS:
            err = abs(float(np.mean(sub_truth <= inv_cdf_col(sub_band, BAND_P, r / 100))) - r / 100) * 100
            if err > worst:
                worst, at_rung = err, r
        recs.append(dict(regime=label, bucket=str(bucket), n=len(idx),
                         worst_err_pp=round(worst, 2), at_rung=at_rung))
    return pd.DataFrame(recs)

regime_tables = {}
for name, df in eval_bands.items():
    tbl = pd.concat([worst_rung_by_bucket(df, df.mid, "magnitude"),       # forecast magnitude
                     worst_rung_by_bucket(df, df.q90 - df.q10, "band_width")],
                    ignore_index=True)
    regime_tables[name] = tbl
    print(f"\n=== {name}: worst rung error within regime buckets ===")
    print(tbl.to_string(index=False))


=== tmax: worst rung error within regime buckets ===
    regime           bucket      n  worst_err_pp  at_rung
 magnitude (-15.491, 22.48] 158923         2.370       20
 magnitude    (22.48, 27.7] 159157         2.460       20
 magnitude     (27.7, 30.3] 158440         2.810       80
 magnitude    (30.3, 33.04] 158835         2.770       20
 magnitude   (33.04, 50.84] 158802         2.590       35
band_width    (0.419, 2.06] 160615         2.720       20
band_width     (2.06, 2.34] 161788         2.520       20
band_width     (2.34, 2.56] 154141         3.670       35
band_width     (2.56, 2.83] 159698         2.990       30
band_width     (2.83, 8.94] 157915         1.190       90



=== tmin: worst rung error within regime buckets ===
    regime           bucket      n  worst_err_pp  at_rung
 magnitude (-22.061, 13.12] 159011         2.600       35
 magnitude   (13.12, 18.78] 158666         2.920       20
 magnitude   (18.78, 22.83] 159375         2.430       20
 magnitude    (22.83, 25.1] 158618         3.410       65
 magnitude    (25.1, 36.71] 158487         3.320       65
band_width    (0.739, 1.61] 159379         5.980       65
band_width     (1.61, 2.13] 158570         3.780       65
band_width     (2.13, 2.57] 161347         2.410       20
band_width     (2.57, 3.11] 156209         3.350       35
band_width     (3.11, 9.05] 158652         3.340       35



=== precip: worst rung error within regime buckets ===
    regime          bucket      n  worst_err_pp  at_rung
 magnitude  (-0.001, 4.89] 635444        67.080        5
 magnitude  (4.89, 408.86] 158713         4.580       65
band_width  (-0.001, 1.52] 317725        88.420        5
band_width     (1.52, 4.3] 158825        71.120        5
band_width    (4.3, 13.83] 158858        19.860        5
band_width (13.83, 108.12] 158749         4.260       65



=== wind: worst rung error within regime buckets ===
    regime        bucket      n  worst_err_pp  at_rung
 magnitude (0.559, 2.56] 160222         3.480       65
 magnitude  (2.56, 3.26] 158687         1.660       80
 magnitude  (3.26, 4.01] 158534         1.630       35
 magnitude  (4.01, 5.02] 158104         1.800       20
 magnitude (5.02, 35.63] 158610         4.020       35
band_width (0.179, 1.26] 159262         3.610       65
band_width  (1.26, 1.52] 159279         2.260       80
band_width  (1.52, 1.77] 159179         1.550       35
band_width  (1.77, 2.12] 159613         2.760       40
band_width  (2.12, 6.35] 156824         2.740       35


In [8]:
# --- Gate 1 verdict per variable ------------------------------------------------
GATE_MARGINAL_PP = 2.0     # tmax/tmin/wind: every rung within 2pp marginal
GATE_REGIME_PP   = 4.0     # ... and within 4pp in every n>=5000 regime bucket
GATE_PRECIP_PP   = 4.0     # precip: wet-forecast subset within 4pp

recs = []
for name in ("tmax", "tmin", "wind"):
    marg = rung_tables[name].err_pp.abs().max()
    reg  = regime_tables[name].worst_err_pp.max() if len(regime_tables[name]) else np.nan
    ok = (marg <= GATE_MARGINAL_PP) and (np.isnan(reg) or reg <= GATE_REGIME_PP)
    recs.append(dict(var=name, max_marginal_pp=marg, max_regime_pp=reg,
                     verdict="PASS" if ok else "FAIL"))

# precip: judged on the WET subset; all-rows point-mass rungs are listed, not failed
wet_marg = rung_tables["precip_wet"].err_pp.abs().max()
flat_rungs = rung_tables["precip"].query("flat_frac > 0.5").rung.tolist()
recs.append(dict(var="precip", max_marginal_pp=wet_marg,
                 max_regime_pp=(regime_tables["precip"].worst_err_pp.max()
                                if len(regime_tables["precip"]) else np.nan),
                 verdict="PASS" if wet_marg <= GATE_PRECIP_PP else "FAIL"))

verdict = pd.DataFrame(recs).set_index("var")
print("Gate 1 verdict — max |coverage - rung|, percentage points")
print("(precip max_marginal_pp is the WET subset; regime col shown for context)\n")
print(verdict.to_string())
print(f"\nprecip all-rows rungs on a point-mass segment (flat_frac>0.5), listed not failed: {flat_rungs}")

Gate 1 verdict — max |coverage - rung|, percentage points
(precip max_marginal_pp is the WET subset; regime col shown for context)

        max_marginal_pp  max_regime_pp verdict
var                                           
tmax              2.430          3.670    FAIL
tmin              2.060          5.980    FAIL
wind              1.220          4.020    FAIL
precip            8.930         88.420    FAIL

precip all-rows rungs on a point-mass segment (flat_frac>0.5), listed not failed: [5, 10, 15, 20, 25, 30, 35, 40, 45]


## Reading it — Gate 1 (for the reviewing session)

**What each table is.** `err_pp` = empirical coverage minus the rung, in
percentage points (`+` = the CDF over-covers: the settled value lands *below* the
rung more often than the rung claims). `kind=trained` rungs (5/10/25/50/75/90/95) are
Level-0 sanity anchors — they should roughly reproduce
`cqr_per_level_validity.ipynb` (here they are re-measured out-of-sample on the
eval half, so expect a little more noise). As of the q9 retrain that anchor set
now includes **25 and 75**, which used to be probit shoulders. The
`kind=interp` rungs (15,20,30,…,85) are the news: all are linear interpolations
*between* the 9 trained points. `flat_frac` = fraction of rows whose
bracketing CDF segment is degenerate (a point mass) — precip only.

**Gate 1 — PASS if:**

- **tmax / tmin / wind** — every rung `|err_pp| ≤ 2` marginal, **and** every
  regime bucket (magnitude- or width-quintile, `n ≥ 5000`) worst rung `≤ 4 pp`.
- **precip** — the **wet-forecast subset** (`hres ≥ 1mm`) within **4 pp**;
  all-rows deviations that sit on a point-mass segment (the listed
  `flat_frac > 0.5` rungs) are the known zero-inflation artifact — they are
  **listed, not failed**.

**FAIL → stop.** Notebook 2 is pointless until this passes. Report which
rungs/regimes miss and by how much, and recommend one of: add trained heads at the
failing rungs, recalibrate the CQR shifts, or coarsen the card's 5% rounding.

**Comparison to the 7-level run.** The rungs between 10 and 90 should improve or
hold: two of the points bounding them are now fitted rather than assumed
normal-ish. A *degradation* there points at the retrain, not the interpolation.

**Scope caveat.** Bands here are the raw ML-table path for *all* cells, including
the small gated set (`gated` column: tmax/tmin/wind ≪ 1%, precip a few % of eval
rows) whose production band uses a different, already-validated empirical
mechanism. Gating does not move the temp/wind read; for precip it is folded into
the dry-day framing above. **Notebook 2 excludes gated cells** from the
end-to-end claim check.